# Chapter 19
## Bursting
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter19.ipynb)

## About this chapter

Bursting adds a slow potassium current on top of a fast spiking model, so a
single slow gate carries the trajectory back and forth between a resting
attractor and a spiking attractor of the fast subsystem: while spiking,
$n_{\rm slow}$ rises and eventually shuts spiking off; while silent, it
decays and eventually lets spiking resume. Two fast subsystems are used --
INaP+I$_K$ (2D, silent/spiking separated by a SNIC) and Erisir (silent/spiking
via a subcritical Hopf-like transition) -- so the same slow-feedback idea
plays out with two different fast-subsystem geometries. Weak/reference/strong
variants of the slow conductance $g_{K,\rm slow}$ show how the burst duty
cycle depends on the feedback strength; the `SHOW_SLOW_I_K` and `ELLIPSES`
variants annotate the trace to make the underlying threshold-crossing
mechanism visible; `INAPIK_PLUS_SLOW_I_K_3D` plots one settled burst cycle in
$(v,n,n_{\rm slow})$ space.

See [`README.md`](chapter19.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact

## INaP+I$_K$ Gating (shared by the `INAPIK_*` examples below)

In [ ]:
def m_inf(v):
    return 1.0 / (1 + exp((-20 - v) / 15))


def n_inf(v):
    return 1.0 / (1 + exp((-25 - v) / 5))


def n_slow_inf(v):
    return 1.0 / (1 + exp((-20 - v) / 5))

## INaP+I$_K$ Bursting with Slow Potassium Feedback

Reference slow conductance $g_{K,\rm slow}=5$; try the slider below to see
weaker/stronger feedback change the burst duty cycle.

In [ ]:
def simulate_inapik_plus_slow_i_k(c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                                   v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15,
                                   i_ext=7.0, g_k_slow=5.0, tau_n_slow=20.0,
                                   t_final=100.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    n_slow = np.zeros(m_steps + 1)
    v[0], m[0], n[0], n_slow[0] = -70.0, m_inf(-70.0), 0.6, 0.0

    for k in range(m_steps):
        v_inc = (g_na * m[k] * (v_na - v[k]) + g_k * n[k] * (v_k - v[k])
                 + g_k_slow * n_slow[k] * (v_k - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = (n_inf(v[k]) - n[k]) / tau_n
        n_slow_inc = (n_slow_inf(v[k]) - n_slow[k]) / tau_n_slow

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        n_tmp = n[k] + dt05 * n_inc
        n_slow_tmp = n_slow[k] + dt05 * n_slow_inc

        v_inc = (g_na * m_tmp * (v_na - v_tmp) + g_k * n_tmp * (v_k - v_tmp)
                 + g_k_slow * n_slow_tmp * (v_k - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = (n_inf(v_tmp) - n_tmp) / tau_n
        n_slow_inc = (n_slow_inf(v_tmp) - n_slow_tmp) / tau_n_slow

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        n[k + 1] = n[k] + dt * n_inc
        n_slow[k + 1] = n_slow[k] + dt * n_slow_inc

    t = np.arange(m_steps + 1) * dt
    return t, v


def plot_inapik_plus_slow_i_k(t, v):
    plt.figure(figsize=(7, 3.5))
    plt.plot(t, v, '-k', linewidth=2)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_inapik_plus_slow_i_k(*simulate_inapik_plus_slow_i_k())

In [ ]:
interact(lambda g_k_slow=5.0, tau_n_slow=20.0:
          plot_inapik_plus_slow_i_k(*simulate_inapik_plus_slow_i_k(
              g_k_slow=g_k_slow, tau_n_slow=tau_n_slow)))

## INaP+I$_K$ Bursting: Weak Slow Feedback

Same as above with $g_{K,\rm slow}=4$ -- barely enough feedback to end a burst.

In [ ]:
def simulate_inapik_plus_weak_slow_i_k(**kwargs):
    kwargs.setdefault('g_k_slow', 4.0)
    return simulate_inapik_plus_slow_i_k(**kwargs)

In [ ]:
plot_inapik_plus_slow_i_k(*simulate_inapik_plus_weak_slow_i_k())

## INaP+I$_K$ Bursting: Strong Slow Feedback

$g_{K,\rm slow}=20$ -- feedback strong enough to shorten each burst sharply.

In [ ]:
def simulate_inapik_plus_strong_slow_i_k(**kwargs):
    kwargs.setdefault('g_k_slow', 20.0)
    return simulate_inapik_plus_slow_i_k(**kwargs)

In [ ]:
plot_inapik_plus_slow_i_k(*simulate_inapik_plus_strong_slow_i_k())

## INaP+I$_K$ Bursting: Effective Drive and Thresholds

Plots $I_{\rm eff}=I+g_{K,\rm slow}n_{\rm slow}(v_K-v)$ alongside the
voltage trace, with the SNIC onset/offset currents marked -- a burst ends
once $I_{\rm eff}$ drops below the lower line and begins again once it
rises back above the upper line.

In [ ]:
def simulate_inapik_show_slow_i_k(c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                                   v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15,
                                   i_ext=7.0, g_k_slow=5.0, tau_n_slow=20.0,
                                   t_final=100.0, dt=0.01):
    t, v = simulate_inapik_plus_slow_i_k(
        c=c, g_na=g_na, g_k=g_k, g_l=g_l, v_na=v_na, v_k=v_k, v_l=v_l, tau_n=tau_n,
        i_ext=i_ext, g_k_slow=g_k_slow, tau_n_slow=tau_n_slow, t_final=t_final, dt=dt)

    # recompute n_slow alongside v to build the effective-drive trace
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    m = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    n_slow = np.zeros(m_steps + 1)
    m[0], n[0], n_slow[0] = m_inf(-70.0), 0.6, 0.0
    for k in range(m_steps):
        v_inc = (g_na * m[k] * (v_na - v[k]) + g_k * n[k] * (v_k - v[k])
                 + g_k_slow * n_slow[k] * (v_k - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = (n_inf(v[k]) - n[k]) / tau_n
        n_slow_inc = (n_slow_inf(v[k]) - n_slow[k]) / tau_n_slow

        v_tmp = v[k] + dt05 * v_inc
        n_tmp = n[k] + dt05 * n_inc
        n_slow_tmp = n_slow[k] + dt05 * n_slow_inc

        n_inc = (n_inf(v_tmp) - n_tmp) / tau_n
        n_slow_inc = (n_slow_inf(v_tmp) - n_slow_tmp) / tau_n_slow

        m[k + 1] = m_inf(v[k + 1])
        n[k + 1] = n[k] + dt * n_inc
        n_slow[k + 1] = n_slow[k] + dt * n_slow_inc

    i_eff = i_ext + g_k_slow * n_slow * (v_k - v)
    return t, v, n_slow, i_eff


def plot_inapik_show_slow_i_k(t, v, n_slow, i_eff):
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, v, '-k', linewidth=2)
    ax[0].set_ylabel('$v$ [mV]')
    ax[0].set_xlim(40, 90)
    ax[0].set_ylim(-80, 0)

    ax[1].plot(t, i_eff, '-k', linewidth=2)
    ax[1].plot([40, 90], [4.5, 4.5], '-r', linewidth=2)
    ax[1].plot([40, 90], [-1.4, -1.4], '-b', linewidth=2)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel(r'$I_{\rm eff}$ [$\mu$A/cm$^2$]')
    ax[1].set_xlim(40, 90)
    ax[1].set_ylim(-2, 7)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_inapik_show_slow_i_k(*simulate_inapik_show_slow_i_k())

## INaP+I$_K$ Square Waves

Highlights the quasi-steady (slow) portions of the trace, where the
central-difference $dv/dt$ is small -- the fast subsystem spends most of
its time near one of the two branches, switching quickly between them.

In [ ]:
def simulate_square_waves(**kwargs):
    return simulate_inapik_plus_slow_i_k(**kwargs)


def plot_square_waves(t, v, dt=0.01):
    plt.figure(figsize=(7, 3.5))
    plt.plot(t, v, '-k', linewidth=1)

    v_left, v_right = v[:-2], v[2:]
    ind = np.where(np.abs(v_right - v_left) / dt < 1)[0]
    plt.plot(t[ind + 1], v[ind + 1], '-r', linewidth=2)

    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_square_waves(*simulate_square_waves())

## INaP+I$_K$ Burst Cycle in 3D

One settled burst cycle plotted in $(v,n,n_{\rm slow})$ space -- the loop
traces the fast spiking/resting transitions while $n_{\rm slow}$ slowly
rises and falls.

In [ ]:
def simulate_inapik_plus_slow_i_k_3d(c=1.0, g_na=20.0, g_k=10.0, g_k_slow=5.0,
                                      g_l=8.0, v_na=60.0, v_k=-90.0, v_l=-80.0,
                                      tau_n=0.15, tau_n_slow=20.0, i_ext=7.0,
                                      t_final=1500.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    n_slow = np.zeros(m_steps + 1)
    v[0], m[0], n[0], n_slow[0] = -70.0, m_inf(-70.0), 0.6, 0.0

    k_vec = []
    for k in range(m_steps):
        v_inc = (g_na * m[k] * (v_na - v[k]) + g_k * n[k] * (v_k - v[k])
                 + g_k_slow * n_slow[k] * (v_k - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = (n_inf(v[k]) - n[k]) / tau_n
        n_slow_inc = (n_slow_inf(v[k]) - n_slow[k]) / tau_n_slow

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        n_tmp = n[k] + dt05 * n_inc
        n_slow_tmp = n_slow[k] + dt05 * n_slow_inc

        v_inc = (g_na * m_tmp * (v_na - v_tmp) + g_k * n_tmp * (v_k - v_tmp)
                 + g_k_slow * n_slow_tmp * (v_k - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = (n_inf(v_tmp) - n_tmp) / tau_n
        n_slow_inc = (n_slow_inf(v_tmp) - n_slow_tmp) / tau_n_slow

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        n[k + 1] = n[k] + dt * n_inc
        n_slow[k + 1] = n_slow[k] + dt * n_slow_inc

        # a full pass around the limit cycle is complete each time n_slow
        # drops back below 0.02
        if n_slow[k] > 0.02 and n_slow[k + 1] <= 0.02:
            k_vec.append(k)

    # only the last full passage through the limit cycle, so the plot shows
    # the cycle itself but not the transient approach to it
    k1, k2 = k_vec[-2], k_vec[-1]
    v_cyc = v[k1:k2 + 1]
    n_cyc = n[k1:k2 + 1]
    n_slow_cyc = n_slow[k1:k2 + 1]
    return v_cyc, n_cyc, n_slow_cyc


def plot_inapik_plus_slow_i_k_3d(v_cyc, n_cyc, n_slow_cyc):
    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(projection='3d')
    ax.plot(v_cyc, n_cyc, n_slow_cyc, '-k', linewidth=2)
    ax.set_xlabel('$v$')
    ax.set_ylabel('$n$')
    ax.set_zlabel(r'$n_{\rm slow}$')
    ax.set_xlim(-70, 0)
    ax.set_ylim(0, 0.7)
    ax.set_zlim(0, 0.06)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_inapik_plus_slow_i_k_3d(*simulate_inapik_plus_slow_i_k_3d())

## Erisir Gating (shared by the `ERISIR_*`/`ELLIPSES` examples below)

In [ ]:
def alpha_h(v):
    return 0.0035 / exp(v / 24.186)


def alpha_m(v):
    return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)


def alpha_n(v):
    return (95 - v) / (exp((95 - v) / 11.8) - 1)


def beta_h(v):
    return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)


def beta_m(v):
    return 1.2262 / exp(v / 42.248)


def beta_n(v):
    return 0.025 / exp(v / 22.222)


def erisir_m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def erisir_h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def erisir_n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))

## Erisir Bursting with Slow Potassium Feedback

In [ ]:
def simulate_erisir_plus_slow_i_k(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5,
                                   v_k=-90.0, v_na=60.0, v_l=-70.0,
                                   i_ext=7.5, g_k_slow=1.5, tau_n_slow=100.0,
                                   t_final=1000.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    n_slow = np.zeros(m_steps + 1)
    v[0] = -70.0
    m[0] = erisir_m_inf(v[0])
    h[0] = erisir_h_inf(v[0])
    n[0] = erisir_n_inf(v[0])
    n_slow[0] = n_slow_inf(v[0])

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 2 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k])
                 + g_k_slow * n_slow[k] * (v_k - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]
        n_slow_inc = (n_slow_inf(v[k]) - n_slow[k]) / tau_n_slow

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = erisir_m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        n_slow_tmp = n_slow[k] + dt05 * n_slow_inc

        v_inc = (g_k * n_tmp ** 2 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_k_slow * n_slow_tmp * (v_k - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        n_slow_inc = (n_slow_inf(v_tmp) - n_slow_tmp) / tau_n_slow

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = erisir_m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        n_slow[k + 1] = n_slow[k] + dt * n_slow_inc

    t = np.arange(m_steps + 1) * dt
    return t, v, n_slow


def plot_erisir_plus_slow_i_k(t, v, n_slow):
    plt.figure(figsize=(7, 4))
    plt.plot(t, v, '-k', linewidth=2)
    plt.xlim(0, t[-1])
    plt.ylim(-95, 55)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_erisir_plus_slow_i_k(*simulate_erisir_plus_slow_i_k())

## Erisir Bursting: Effective Drive and Thresholds

In [ ]:
def plot_erisir_show_slow_i_k(t, v, n_slow, i_ext=7.5, g_k_slow=1.5, v_k=-90.0):
    i_eff = i_ext + g_k_slow * n_slow * (v_k - v)
    t_final = t[-1]

    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, v, '-k', linewidth=2)
    ax[0].set_ylabel('$v$ [mV]')
    ax[0].set_xlim(200, 600)
    ax[0].set_ylim(-95, 55)

    ax[1].plot(t, i_eff, '-k', linewidth=2)
    ax[1].plot([0, t_final], [7.025, 7.025], '-r', linewidth=2)
    ax[1].plot([0, t_final], [6.475, 6.475], '-b', linewidth=2)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel(r'$I_{\rm eff}$ [$\mu$A/cm$^2$]')
    ax[1].set_xlim(200, 600)
    ax[1].set_ylim(6.3, 7.5)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_erisir_show_slow_i_k(*simulate_erisir_plus_slow_i_k())

## Erisir Bursting: Local Extrema Marked

Marks the local maxima and minima of the voltage trace in red, tracing an
ellipse-like envelope around each spike.

In [ ]:
def plot_ellipses(t, v, n_slow):
    t_final = t[-1]
    plt.figure(figsize=(7, 4))
    plt.plot(t, v, '-k', linewidth=1)

    v_left, v_center, v_right = v[:-2], v[1:-1], v[2:]
    ind = np.where((v_center > v_left) & (v_center > v_right))[0]
    plt.plot(t[ind + 1], v[ind + 1], '-r', linewidth=2)
    ind = np.where((v_center < v_left) & (v_center < v_right))[0]
    plt.plot(t[ind + 1], v[ind + 1], '-r', linewidth=2)

    plt.xlim(0, t_final)
    plt.ylim(-95, 55)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_ellipses(*simulate_erisir_plus_slow_i_k())